# FoodShock — Cross-Country ASEAN Food Price Forecasting & Early Shock Detection System

**Machine Learning Solution Challenge (MLSC) – Cyan Academic Competition 2026**  
**Organizer:** Enterprise Data Management (EDM) Laboratory  
**Cabang:** Machine Learning Solution Challenge (EDM Lab)  
**Target SDG:** **SDG 2 — Zero Hunger**  
**Solution Name:** FoodShock  
**Focus:** Residual Forecasting + Early Food-Price Shock Detection across ASEAN

---

## 1. Latar Belakang & Keterkaitan SDG 2 — Zero Hunger

Ketahanan pangan di kawasan ASEAN menghadapi tekanan struktural (perubahan iklim, gangguan rantai pasok, volatilitas kurs) dan tekanan siklikal (musiman panen, permintaan festival). Fluktuasi harga pangan pokok — khususnya beras, daging ayam/sapi/babi, dan minyak goreng — secara langsung memengaruhi daya beli rumah tangga miskin, inflasi pangan, dan stabilitas sosial.

Data WFP/HDX 2015–2025 menunjukkan heterogenitas harga yang tinggi antar negara dan antar jenis pasar (Retail vs Wholesale). Shock harga yang tidak terdeteksi dini mengurangi efektivitas intervensi kebijakan (operasi pasar, pelepasan stok, subsidi tepat sasaran).

**Keterkaitan konkret dengan SDG 2:**
- Target 2.1 & 2.2: akses pangan aman, bergizi, dan terjangkau bagi semua.
- Sistem peringatan dini berbasis machine learning pada level panel antar-negara memberikan instrumen operasional bagi ASEAN Food Security Information System (AFSIS) dan kementerian pertanian/perdagangan nasional.


## Shock Threshold Justification (+5% MoM)

1. **Policy:** Consistent with WFP ALPS (Alert for Price Spikes) and ASEAN AFSIS food-price surveillance: a MoM staple increase above **+5%** indicates acute affordability stress.
2. **Empirical:** On `target_pct_change_t1`, **P90 ≈ +4.44%**, **P95 ≈ +7.20%**, IQR ≈ [−1.89%, +1.77%]. The +5% rule isolates the upper ~9% tail and is paired with adaptive series thresholds and a Stage-2 calibrated probability threshold (≈0.35 for max F1).

## Walk-Forward & Two-Stage Architecture

- **Expanding-window walk-forward** (cutoffs 2019-06 … 2024-06, 12-month test each): all seven countries receive out-of-sample evaluation.
- **Stage 1A:** Residual point forecast from current price $y_t$ (`usdprice`).
- **Stage 1B:** Quantile models (P10/P90) → PICP ≈ 80%.
- **Stage 2:** Balanced LightGBM classifier for $P(\mathrm{Shock}_{t+1})$ with F1-optimal threshold.
- **Econometric honesty:** prices are near-martingale; report **SS_RMSE**, **Diebold–Mariano** tests, and classifier metrics rather than level $R^2$ alone.


## 2. Audit Cakupan Data ASEAN (2015–2025) & Standardisasi

### Aturan yang diterapkan
- Hanya seri (`countryiso3` + `market` + `commodity_group` + `pricetype` + `priceflag` + `unit`) dengan **≥ 60 bulan** observasi.
- 7 negara ASEAN: IDN, KHM, LAO, PHL, THA, TLS, MMR.
- 5 kelompok komoditas baku yang didukung minimal 3 negara: **Rice, Chicken Meat, Beef Meat, Pork Meat, Cooking Oil**.
- Harga dalam **USD**; unit distandardisasi (KG untuk beras/daging, L untuk minyak; konversi 5 L dan KG minyak dengan densitas ≈ 0.92).
- Pemisahan eksplisit `pricetype` (Retail / Wholesale) dan `priceflag`.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

country_cov = pd.read_csv('../data/fpma_wfp_country_coverage.csv')
com_cov = pd.read_csv('../data/fpma_wfp_commodity_coverage.csv')
print("=== Country Coverage (series ≥ 60 months) ===")
display(country_cov)
print("\n=== Commodity Group Coverage ===")
display(com_cov)


=== Country Coverage (series ≥ 60 months) ===
  countryiso3  n_series  n_commodity_groups  avg_months  max_months    min_date    max_date
0         PHL       305                   4     68.8262         116  2015-01-15  2025-12-15
1         IDN       100                   4     60.0800          62  2015-01-15  2024-05-15
2         LAO        82                   5    111.1707         127  2015-01-15  2025-12-15
3         KHM        69                   3     69.2609          88  2015-01-15  2025-12-15
4         MMR        17                   2     77.3529         105  2015-01-15  2025-12-15
5         TLS         7                   3     71.1429          94  2015-01-15  2025-12-15
6         THA         2                   2     63.0000          63  2015-01-15  2020-03-15

=== Commodity Group Coverage ===
  commodity_group  n_series  n_countries                    countries  avg_months  avg_price_usd
0            Rice       204            7  IDN,KHM,LAO,MMR,PHL,THA,TLS     71.1127      

**Catatan penting mengenai IDN & THA:**

- **IDN** seri berakhir sekitar Mei 2024; **THA** sekitar Maret 2020.
- Pada skema **expanding-window walk-forward** (cutoff 2019–2024), keduanya **tetap masuk evaluasi out-of-sample** pada fold di mana data uji masih tersedia (mis. THA pada fold 2019-06; IDN hingga fold sebelum data habis).
- Fold terakhir (2024-06) hanya mencakup negara dengan observasi setelah Juni 2024 (KHM, LAO, MMR, PHL, TLS) — konsisten dengan ketersediaan data, bukan eksklusi sewenang-wenang.


## 3. Calendar-Safe Feature Engineering (Zero Data Leakage)

### Prinsip
- Re-indexing setiap seri ke kalender bulanan penuh **tanpa forward-fill harga mentah**.
- Semua lag dan rolling statistics dihitung dari `shift(1)` sehingga fitur pada waktu *t* hanya menggunakan informasi hingga *t−1*.
- Target residual: $\Delta y_{t+1} = y_{t+1} - y_t$ (menghilangkan bias level antar komoditas yang memiliki skala harga sangat berbeda).
- Rekonstruksi prediksi akhir: $\hat{y}_{t+1} = y_t + \widehat{\Delta y}_{t+1}$.

### Fitur yang dihasilkan
- Lags: `lag_1`, `lag_2`, `lag_3`, `lag_6`, `lag_12`
- Rolling (dari shift(1)): `rolling_mean_3/6`, `rolling_std_6`, `rolling_min_6`, `rolling_max_6`, `price_to_mean6`, `cv_6`
- Momentum: `pct_change_1m`, `pct_change_3m`, `pct_change_12m`
- Musiman: `month_sin`, `month_cos`
- Konteks: `countryiso3`, `commodity_group`, `pricetype`, `priceflag`


In [2]:
model_ready = pd.read_csv('../data/foodshock_asean_model_ready.csv')
model_ready['date'] = pd.to_datetime(model_ready['date'])
print("Shape:", model_ready.shape)
print("Date range:", model_ready['date'].min().date(), "→", model_ready['date'].max().date())
print("Countries:", sorted(model_ready['countryiso3'].unique()))
print("Commodity groups:", sorted(model_ready['commodity_group'].unique()))
print("\nSample of engineered features:")
display(model_ready[['date','countryiso3','commodity_group','usdprice','lag_1','lag_12',
                     'rolling_mean_6','cv_6','pct_change_1m','month_sin',
                     'target_price_usd_t1']].head(6))


Shape: (27950, 38)
Date range: 2016-02-01 → 2025-11-01
Countries: ['IDN', 'KHM', 'LAO', 'MMR', 'PHL', 'THA', 'TLS']
Commodity groups: ['Beef Meat', 'Chicken Meat', 'Cooking Oil', 'Pork Meat', 'Rice']

Sample of engineered features:


,date,countryiso3,commodity_group,usdprice,lag_1,lag_12,rolling_mean_6,cv_6,pct_change_1m,month_sin,target_price_usd_t1
0,2016-02-01,IDN,Beef Meat,7.8700,7.5000,7.4300,7.3717,0.0242,0.0331,0.8660,8.0200
1,2016-02-01,IDN,Chicken Meat,2.9800,2.9800,2.8000,2.7783,0.0636,0.0876,0.8660,2.7500
2,2016-02-01,IDN,Cooking Oil,1.0000,0.9600,1.0900,0.9700,0.0261,0.0105,0.8660,1.0200
3,2016-02-01,IDN,Rice,1.0000,0.9600,1.0000,0.9417,0.0288,0.0213,0.8660,1.0100
4,2016-02-01,KHM,Rice,0.3500,0.3500,0.3200,0.3717,0.0575,0.0000,0.8660,0.3500
5,2016-02-01,KHM,Rice,0.2700,0.3200,0.3000,0.3333,0.0727,-0.0588,0.8660,0.3000


### Tren Harga Rata-rata per Kelompok Komoditas
![EDA Trends](../figures/eda_trends.png)


## 4. Pelatihan & Evaluasi Residual Forecasting (V2 Walk-Forward)

### Validasi
**Expanding-window walk-forward** dengan 6 fold tahunan (cutoff: 2019-06 … 2024-06; tiap fold menguji 12 bulan ke depan). Seluruh **7 negara ASEAN** (termasuk IDN & THA pada fold awal) dievaluasi secara *out-of-sample*. Metrik dilaporkan **micro-averaged** (seluruh observasi) dan **macro-averaged** (bobot setara antarnegara).

### Model yang dibandingkan
1. **Baseline:** Persistence (\(y_t\) = `usdprice`)
2. **Residual point models (fair Optuna):** Random Forest, XGBoost, LightGBM, CatBoost, HistGradientBoosting — target \(\Delta y_{t+1} = y_{t+1} - y_t\), rekonstruksi \(\hat{y}_{t+1} = y_t + \widehat{\Delta y}_{t+1}\)
3. **Stage 1B:** Quantile residual models (P10 & P90) → **PICP ≈ 80.3%**
4. **Stage 2:** LightGBMClassifier (`class_weight="balanced"`) untuk \(P(\mathrm{Shock}_{t+1})\) dengan ambang probabilitas optimal F1 ≈ **0.35**

### Fitur
Lags & rolling calendar-safe + **implied FX** (`fx_pct_1m`, `fx_pct_3m`, `fx_vol_6m`) + **ASEAN spillover** (`asean_comm_mom_1m`, `country_vs_asean_mom`).


In [3]:
results = pd.read_csv('../data/model_evaluation_results.csv')
print("=== Walk-Forward Model Comparison (Residual Reconstruction) ===")
sort_col = 'MAE_micro' if 'MAE_micro' in results.columns else ('MAE' if 'MAE' in results.columns else results.columns[1])
display(results.sort_values(sort_col))
print("\nColumns:", list(results.columns))


=== Walk-Forward Model Comparison (Residual Reconstruction) ===

Columns: ['model', 'MAE_micro', 'RMSE_micro', 'sMAPE_micro', 'R2_micro', 'MAE_macro', 'Persistence_MAE', 'Persistence_RMSE', 'SS_RMSE', 'DM_stat', 'DM_pvalue']


,model,MAE_micro,RMSE_micro,sMAPE_micro,R2_micro,MAE_macro,Persistence_MAE,Persistence_RMSE,SS_RMSE,DM_stat,DM_pvalue
1,CatBoost_WF_Residual,0.0984,0.1931,3.0611,0.9937,0.0978,0.0986,0.1950,0.0102,-6.0076,0.0000
3,Persistence (y_t),0.0986,0.1950,3.0411,0.9936,NaN,0.0986,0.1950,0.0000,NaN,NaN
0,XGBoost_WF_Residual,0.0995,0.1930,3.0854,0.9937,0.0995,0.0986,0.1950,0.0104,-2.0488,0.0405
2,LightGBM_WF_Residual,0.0997,0.1937,3.1438,0.9937,0.0998,0.0986,0.1950,0.0070,-4.0768,0.0000
4,HistGradientBoosting_WF_Residual,0.1015,0.1952,3.1789,0.9936,0.1003,0.0986,0.1950,-0.0007,0.2543,0.7993
5,RandomForest_WF_Residual,0.1017,0.1971,3.1375,0.9934,0.1013,0.0986,0.1950,-0.0104,3.3807,0.0007


### Ablation Study: Direct Level Forecasting vs. Residual Forecasting

Prediksi level absolut mencampur skala harga (Rice ≈ USD 0.8 vs Beef ≈ USD 6–7) sehingga tree-based model bias terhadap komoditas mahal. **Residual forecasting** dari \(y_t\) (`usdprice`) menormalkan target ke perubahan harga.

| Approach | Temuan V2 (Walk-Forward) |
|----------|--------------------------|
| Persistence (\(y_t\)) | MAE_micro ≈ 0.0986 · RMSE_micro ≈ 0.1950 |
| **Best residual ML (XGBoost / CatBoost)** | RMSE_micro ≈ **0.1930–0.1931** · **SS_RMSE > 0** · **DM p-value signifikan** (CatBoost \(p < 0.001\), XGBoost \(p \approx 0.04\)) |

Harga pangan bulanan bersifat *near-martingale*: nilai tambah utama sistem bukan selisih MAE yang besar, melainkan **skill RMSE**, **signifikansi Diebold–Mariano**, **interval kuantil P10–P90 (PICP ≈ 80.3%)**, dan **deteksi dini probabilistik**.


### Visualisasi Perbandingan Model & Actual vs Predicted
![Model Comparison](../figures/model_comparison.png)
![Actual vs Predicted](../figures/actual_vs_predicted.png)


## 5. Evaluasi Terpisah per Komoditas, Negara, dan Jenis Pasar


In [4]:
print("=== By Commodity Group ===")
display(pd.read_csv('../data/model_eval_by_commodity.csv'))
print("\n=== By Country ===")
display(pd.read_csv('../data/model_eval_by_country.csv'))
print("\n=== By Price Type (Retail vs Wholesale) ===")
display(pd.read_csv('../data/model_eval_by_pricetype.csv'))


=== By Commodity Group ===

=== By Country ===

=== By Price Type (Retail vs Wholesale) ===


,commodity_group,MAE,RMSE,sMAPE,R2,n
0,Beef Meat,0.1546,0.2503,2.3839,0.9533,4994.0000
1,Chicken Meat,0.1264,0.2282,3.8252,0.8552,4186.0000
2,Cooking Oil,0.0659,0.1993,4.3632,0.7966,1709.0000
3,Pork Meat,0.1554,0.2349,3.1134,0.9311,5069.0000
4,Rice,0.0221,0.0384,2.8435,0.9514,7933.0000


,countryiso3,MAE,RMSE,sMAPE,R2,n
0,IDN,0.0670,0.1437,2.9630,0.9974,3780.0000
1,KHM,0.0786,0.1519,4.0187,0.9921,2629.0000
2,LAO,0.1245,0.3091,4.0058,0.9752,3185.0000
3,MMR,0.1504,0.7178,8.4849,0.5874,84.0000
4,PHL,0.1057,0.1667,2.6094,0.9947,14013.0000
5,THA,0.0304,0.0442,3.4208,0.9946,18.0000
6,TLS,0.1400,0.2798,10.1639,0.9367,182.0000


,pricetype,MAE,RMSE,sMAPE,R2,n
0,Retail,0.1010,0.1953,3.0173,0.9936,22506.0000
1,Wholesale,0.0757,0.1501,4.1923,0.9915,1385.0000


**Interpretasi (selaras output Cell 13 — evaluasi walk-forward OOS):**

- **Per jenis pasar:** Retail mendominasi sampel uji (*n* = 22.506) dengan MAE ≈ 0,101; Wholesale lebih kecil (*n* = 1.385) namun MAE lebih rendah (≈ 0,076).
- **Per komoditas:** **Rice** dan **Cooking Oil** memiliki error absolut terendah (harga relatif stabil); **Pork Meat** dan **Beef Meat** lebih volatil sehingga MAE lebih tinggi.
- **Per negara (7 ASEAN, seluruhnya OOS pada fold yang relevan):** PHL mendominasi jumlah observasi uji; **IDN** (*n* ≈ 3.780) dan **THA** (*n* ≈ 18) ikut dievaluasi *out-of-sample* pada fold di mana data uji tersedia — bukan hanya tersimpan di training set. MMR/TLS memiliki sampel tipis sehingga metrik lebih bervariasi.
- Secara keseluruhan, residual walk-forward menjaga akurasi lintas segmen tanpa bias skala antar komoditas.


## 6. Interpretabilitas SHAP Values & Deteksi Dini Food Price Shock

### SHAP Summary
![SHAP Summary](../figures/shap_summary.png)

**Key Insights SHAP:**
- Setelah residual modeling, `lag_1` tetap relevan namun kontribusi fitur **momentum** (`pct_change_*`), **volatilitas** (`rolling_std_6`, `cv_6`), dan **musiman** (`month_sin` / `month_cos`) menjadi lebih menonjol.
- Encoding negara menangkap heterogenitas struktural antar pasar ASEAN.


### Food Price Shock Early-Warning Classification

**Definisi label shock (kebijakan + empiris):** \(|\%\Delta_{t+1}| > 5\%\) (selaras WFP ALPS / AFSIS; P90 empiris ≈ +4.44%, P95 ≈ +7.20%).

**Stage-2 model:** classifier probabilistik (LightGBM, `class_weight="balanced"`) dilatih walk-forward; ambang keputusan dioptimalkan F1 pada skor OOF (**threshold ≈ 0.35**), bukan rule kaku pada prediksi titik saja.

**Hasil OOF (6-fold):**
| Precision | Recall | F1 | ROC-AUC | PR-AUC | Base rate | PICP P10–P90 |
|-----------|--------|-----|---------|--------|-----------|-------------|
| 0.213 | **0.564** | **0.309** | 0.605 | 0.228 | 0.162 | **80.3%** |

Recall dan F1 jauh di atas baseline non-informatif; alert operasional menggabungkan \(P(\mathrm{Shock})\ge 0.35\) **atau** lonjakan prediksi \(> +5\%\).


In [5]:
shock = pd.read_csv('../data/model_eval_shock_classification.csv')
print("=== Shock Classification Metrics ===")
display(shock)


=== Shock Classification Metrics ===


,Precision,Recall,F1,ROC_AUC,PR_AUC,Base_Rate,Optimal_Threshold,Support_Shock,Support_Total,PICP_P10_P90
0,0.2129,0.5640,0.3092,0.6046,0.2283,0.1619,0.3500,3869,23891,0.8027


![Shock Residual Distribution](../figures/shock_residual_dist.png)

### Limitasi Empiris & Saran Pengembangan
- Harga komoditas bulanan mendekati *random walk*; Persistence tetap kompetitif pada MAE.
- Coverage tidak merata (THA, MMR relatif tipis) membatasi stabilitas metrik makro.
- Belum tersedia kovariat cuaca/energi yang harmonis di ketujuh negara.
- Precision classifier masih moderat (trade-off recall untuk early warning).

**Arah lanjutan:** ensemble kuantil / conformal prediction, fitur eksternal lintas negara, dan threshold cost-sensitive untuk dashboard AFSIS operasional.


## 7. Kesimpulan & Rekomendasi Kebijakan

FoodShock membangun sistem forecasting & early-warning harga pangan ASEAN yang:

1. **Panel bersih & leakage-safe** — audit ≥ 60 bulan, standardisasi USD/KG/L, fitur kalender tanpa *forward-fill* harga, plus **implied FX** dan **ASEAN spillover**.
2. **Validasi jujur** — *expanding-window walk-forward* 6 fold; seluruh 7 negara masuk evaluasi OOS; metrik micro & macro.
3. **Residual + signifikansi statistik** — model boosting residual mengungguli Persistence pada RMSE dengan **Diebold–Mariano signifikan**; **SS_RMSE positif**.
4. **Two-stage risk layer** — interval **P10–P90 (PICP ≈ 80.3%)** dan **Stage-2 shock classifier (Recall ≈ 56.4%, F1 ≈ 0.309)** mendukung peringatan dini selaras ambang **+5% MoM** (WFP ALPS / AFSIS).

**Rekomendasi:** integrasikan skor \(P(\mathrm{Shock})\) dan batas atas P90 ke dashboard pemantauan harga nasional/regional; prioritaskan intervensi pada jalur komoditas dengan volatilitas tinggi (daging, minyak goreng) dan pasar dengan cakupan data memadai.

*FoodShock · MLSC 2026 · SDG 2 — Zero Hunger*
